# NB6 · Bot contrast and the two missing event studies

Two things the paper needs that the pipeline did not persist.

**1 · The naive-vs-corrected bot table.** Only the Dependabot number survived from the
pre-correction run. This rebuilds the *raw* (bot-inclusive) outcomes on the exact
balanced panel that produced the published results, and re-estimates all eight
outcomes under both TWFE and Callaway–Sant'Anna. Same sample, same cohorts, same
estimator — the only thing that changes is whether bot activity is counted.

**2 · Event studies for `o2_comments_per_pr` and `o6_direct_push_share`.** NB5 kept
the `att_gt` result for only three outcomes, so `event_study.csv` is missing two of
the four Benjamini–Hochberg survivors.

Nothing here re-mines anything. It reads `nb4_panel_balanced.parquet` and
`nb3_panel.parquet`, both already on Drive.

**Runtime** ~30 minutes. Every fit is checkpointed individually — if the runtime
drops, re-running the cell resumes at the next unfinished fit.

In [ ]:
#@title Configuration { display-mode: "form" }
E_MIN    = -12   #@param {type:"integer"}
E_MAX    = 24    #@param {type:"integer"}
N_BOOT   = 500   #@param {type:"integer"}
MIN_CELL = 10    #@param {type:"integer"}
MIN_PRS  = 1     #@param {type:"integer"}
SEED     = 20261005 #@param {type:"integer"}
PRIMARY  = "o1_reviewers_per_pr"

import os, sys, json, gc, hashlib
from pathlib import Path
import numpy as np, pandas as pd
IN_COLAB=False
try:
    from google.colab import drive; IN_COLAB=True
except ImportError: pass
DRIVE_OK=False
if IN_COLAB:
    try: drive.mount("/content/drive",force_remount=False); DRIVE_OK=True
    except Exception as e: print("!! Drive mount FAILED:",e)
DRIVE=("/content/drive/MyDrive/peltzman" if DRIVE_OK else
       "/content/peltzman_LOCAL" if IN_COLAB else os.path.expanduser("~/peltzman_out"))
STATE=Path(DRIVE,"state"); FIG=Path(DRIVE,"figures"); OUT=Path(DRIVE,"results")
CODE=Path("/content" if IN_COLAB else "/tmp")
for _p in (STATE,FIG,OUT,CODE): Path(_p).mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(CODE))
print("state  :",STATE)
print("results:",OUT)

In [ ]:
%%writefile /content/peltzman_io.py
"""
Peltzman Protocol - durable, resumable job state.

Guarantee
---------
Kill the process, lose the runtime, close the browser, at ANY instant:
  * no file is ever left half-written,
  * no completed work is ever lost,
  * no work is ever silently duplicated,
  * restarting the cell resumes from the last completed batch.

How
---
The key list is fixed and ordered at job creation, and batch `i` is defined as
a deterministic slice of it. A batch's output is therefore a pure function of
its index, so re-running a batch overwrites its own file and can never
duplicate rows. Completion is represented by the *existence* of that file and
nothing else - there is no separate done-list that can drift out of sync with
the data.

Every file is written to a temporary name, flushed, fsync'd, and then moved
into place with os.replace(), which is atomic. A crash mid-write leaves a
stray .tmp file (harmless, cleaned on next run) and the previous state intact.

Because completion is file existence, the crash window is exactly one batch.
Size batches so one takes a few minutes.
"""
from __future__ import annotations
import json, os, time, hashlib, tempfile, shutil, glob, sys
from pathlib import Path

__all__ = ["Job", "atomic_write_text", "atomic_write_bytes"]


# --------------------------------------------------------------- atomic io --
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())          # no-op on some mounts; harmless
        os.replace(tmp, path)             # atomic within a filesystem
    except BaseException:
        try: os.unlink(tmp)
        except OSError: pass
        raise


def atomic_write_text(path, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def _hash_params(params) -> str:
    blob = json.dumps(params, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:10]


# --------------------------------------------------------------------- job --
class Job:
    """A resumable pass over an ordered list of keys.

    >>> job = Job("nb1_workflows", keys=repos, root=DRIVE, batch_size=25,
    ...           params={"detector": "v3"})
    >>> for i, keys in job.pending():
    ...     job.commit(i, [row for k in keys for row in scan(k)])
    >>> df = job.frame()

    Changing `params` starts a new, separate run directory: results computed
    under different settings are never silently mixed.
    """

    def __init__(self, name, keys, root, batch_size=25, params=None, verbose=True):
        self.name = name
        self.params = dict(params or {})
        self.batch_size = int(batch_size)
        self.verbose = verbose
        # NOTE: the key count is deliberately NOT part of the run identity.
        # If the input list changes at all we want a loud error, not a silent
        # fresh run that looks like "nothing was ever done".
        self.dir = Path(root) / name / _hash_params(
            {"params": self.params, "batch_size": self.batch_size})
        self.shards = self.dir / "shards"
        self.shards.mkdir(parents=True, exist_ok=True)

        keyfile = self.dir / "keys.json"
        if keyfile.exists():
            stored = json.loads(keyfile.read_text())
            if stored != list(keys):
                raise ValueError(
                    f"{self.name}: key list changed since this run started "
                    f"({len(stored)} -> {len(keys)}). Change `params` to start "
                    f"a new run, or restore the original list.")
        else:
            atomic_write_text(keyfile, json.dumps(list(keys)))
            atomic_write_text(self.dir / "meta.json", json.dumps({
                "name": name, "created": time.time(), "n_keys": len(keys),
                "batch_size": self.batch_size, "params": self.params,
                "python": sys.version.split()[0]}, indent=1))

        self.keys = list(keys)
        self._sweep_tmp()
        if verbose:
            d, n = self.n_done_batches(), self.n_batches()
            print(f"[{name}] {self.dir}")
            print(f"[{name}] {d}/{n} batches complete"
                  + (f" — resuming at batch {d}" if 0 < d < n else
                     " — nothing to do" if d == n else " — starting fresh"))

    # ---------------------------------------------------------- internals --
    def _sweep_tmp(self):
        """Remove leftovers from a crash mid-write. They are never valid."""
        for t in glob.glob(str(self.shards / "*.tmp")) + glob.glob(str(self.dir / "*.tmp")):
            try: os.unlink(t)
            except OSError: pass

    def _path(self, i) -> Path:
        return self.shards / f"batch-{i:06d}.jsonl"

    def n_batches(self) -> int:
        return (len(self.keys) + self.batch_size - 1) // self.batch_size

    def batch_keys(self, i):
        return self.keys[i * self.batch_size:(i + 1) * self.batch_size]

    def done_batches(self):
        return {i for i in range(self.n_batches()) if self._path(i).exists()}

    def n_done_batches(self) -> int:
        return len(self.done_batches())

    # ------------------------------------------------------------- driving --
    def pending(self):
        """Yield (batch_index, keys) for every batch not yet committed."""
        done = self.done_batches()
        for i in range(self.n_batches()):
            if i not in done:
                yield i, self.batch_keys(i)

    def commit(self, i, rows):
        """Atomically record this batch's output. Idempotent by construction."""
        payload = "".join(json.dumps(r, separators=(",", ":"), default=str) + "\n"
                          for r in rows)
        atomic_write_text(self._path(i), payload)
        self.beat(i, len(rows))

    def beat(self, i, n_rows=None, note=None):
        atomic_write_text(self.dir / "heartbeat.json", json.dumps({
            "at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "last_batch": i, "rows": n_rows, "note": note,
            "done": self.n_done_batches(), "total": self.n_batches()}, indent=1))

    def run(self, fn, on_error="record", progress_every=1, after_batch=None):
        """Drive the whole job. `fn(key) -> list[dict]`.

        on_error='record'  capture the failure as a row and move on (default)
                 'skip'    drop the key silently
                 'raise'   stop, keeping every committed batch
        after_batch(job, i)  optional hook run after each successful commit,
                 e.g. to replicate state off the runtime. A failing hook never
                 loses the batch - it is already on disk before the hook runs.
        """
        t0, n_before = time.time(), self.n_done_batches()
        for i, keys in self.pending():
            rows = []
            for k in keys:
                try:
                    rows.extend(fn(k) or [])
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if on_error == "raise":
                        raise
                    if on_error == "record":
                        rows.append({"_key": k, "_error": f"{type(e).__name__}: {e}"})
            self.commit(i, rows)
            if after_batch is not None:
                try: after_batch(self, i)
                except Exception as e:
                    print(f"  after_batch hook failed ({type(e).__name__}: {e}) — "
                          f"batch is still committed locally")
            done = self.n_done_batches()
            if self.verbose and (done - n_before) % progress_every == 0:
                el = time.time() - t0
                per = el / max(1, done - n_before)
                left = (self.n_batches() - done) * per
                print(f"  batch {done}/{self.n_batches()}  "
                      f"{per:5.1f}s/batch  ~{left/60:5.1f} min remaining")
        if self.verbose:
            print(f"[{self.name}] complete: {self.n_batches()} batches")

    # ------------------------------------------------------------ reading --
    def rows(self, drop_errors=True):
        """Iterate every committed row. Truncated trailing lines - which an
        atomic write cannot produce, but a copied/synced file might - are
        skipped and counted rather than raising."""
        bad = 0
        for i in sorted(self.done_batches()):
            with open(self._path(i), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        bad += 1
                        continue
                    if drop_errors and isinstance(r, dict) and "_error" in r:
                        continue
                    yield r
        if bad and self.verbose:
            print(f"[{self.name}] warning: skipped {bad} unparseable line(s) — "
                  f"run .verify() to locate")

    def frame(self, drop_errors=True):
        import pandas as pd
        return pd.DataFrame(list(self.rows(drop_errors=drop_errors)))

    def errors(self):
        return [r for r in self.rows(drop_errors=False)
                if isinstance(r, dict) and "_error" in r]

    # -------------------------------------------------------- maintenance --
    def verify(self):
        """Check every committed shard parses. Returns list of bad batch ids."""
        bad = []
        for i in sorted(self.done_batches()):
            try:
                for line in open(self._path(i), encoding="utf-8"):
                    if line.strip():
                        json.loads(line)
            except Exception as e:
                bad.append((i, f"{type(e).__name__}: {e}"))
        if self.verbose:
            print(f"[{self.name}] verify: {self.n_done_batches()} shards, "
                  f"{len(bad)} damaged")
        return bad

    def reset(self, batches=None):
        """Delete shards so they are recomputed. `batches=None` resets damaged
        ones only; pass an iterable of indices, or 'all'."""
        if batches == "all":
            targets = sorted(self.done_batches())
        elif batches is None:
            targets = [i for i, _ in self.verify()]
        else:
            targets = list(batches)
        for i in targets:
            try: os.unlink(self._path(i))
            except OSError: pass
        if self.verbose:
            print(f"[{self.name}] reset {len(targets)} batch(es)")
        return targets

    def retry_errors(self):
        """Re-queue every batch that recorded at least one failure."""
        hit = set()
        for i in sorted(self.done_batches()):
            for line in open(self._path(i), encoding="utf-8"):
                if '"_error"' in line:
                    hit.add(i); break
        return self.reset(hit)

    def export(self, path, fmt="parquet"):
        """Write the whole result set as one file, atomically."""
        df = self.frame()
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_suffix(p.suffix + ".tmp")
        if fmt == "parquet":
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)
        os.replace(tmp, p)
        if self.verbose:
            print(f"[{self.name}] exported {len(df):,} rows -> {p}")
        return p

    def status(self):
        hb = self.dir / "heartbeat.json"
        return {"dir": str(self.dir), "batches": self.n_batches(),
                "done": self.n_done_batches(), "keys": len(self.keys),
                "heartbeat": json.loads(hb.read_text()) if hb.exists() else None}


In [ ]:
%%writefile /content/panel_tools.py
"""
Peltzman Protocol - panel construction, fast two-way fixed-effects DiD, and
randomisation-based power.

The estimator here is deliberately simple: it exists to check pre-trends and to
measure how much noise the design has, not to produce the paper's numbers.
Callaway-Sant'Anna in NB5 does that. But a fast TWFE fit is what makes a
permutation study affordable, and a permutation study is a far more honest
account of power than plugging a guessed standard deviation into a formula.
"""
from __future__ import annotations
import numpy as np

__all__ = ["month_index", "demean_two_way", "twfe_did", "event_study",
           "placebo_se", "mde_from_se"]


def month_index(ym):
    """'YYYY-MM' -> integer month, so arithmetic and sorting are trivial."""
    y, m = str(ym).split("-")
    return int(y) * 12 + int(m) - 1


def demean_two_way(y, unit, time, iters=30, tol=1e-10):
    """Absorb unit and time fixed effects by alternating projections.

    Equivalent to residualising on the full dummy set, but O(n) per sweep
    instead of building a matrix with tens of thousands of columns. Converges
    geometrically; 30 sweeps is far past what a panel this shape needs.
    """
    y = np.asarray(y, dtype=np.float64).copy()
    nu, nt = unit.max() + 1, time.max() + 1
    cu, ct = np.bincount(unit, minlength=nu), np.bincount(time, minlength=nt)
    cu[cu == 0] = 1; ct[ct == 0] = 1
    for _ in range(iters):
        prev = y[:1000].copy() if y.size >= 1000 else y.copy()
        y -= (np.bincount(unit, y, minlength=nu) / cu)[unit]
        y -= (np.bincount(time, y, minlength=nt) / ct)[time]
        cur = y[:1000] if y.size >= 1000 else y
        if np.max(np.abs(cur - prev)) < tol:
            break
    return y


def twfe_did(y, treat, unit, time, cluster=None):
    """Two-way fixed-effects DiD. Returns (beta, se, n).

    `treat` is the post-treatment indicator. Standard errors are clustered on
    `cluster` (default: unit), because repo-months within a repo are anything
    but independent and unclustered errors here would be fiction.
    """
    y = np.asarray(y, float); d = np.asarray(treat, float)
    keep = np.isfinite(y) & np.isfinite(d)
    y, d, unit, time = y[keep], d[keep], unit[keep], time[keep]
    if y.size < 10 or d.sum() < 2 or (1 - d).sum() < 2:
        return np.nan, np.nan, int(y.size)
    yt = demean_two_way(y, unit, time)
    dt = demean_two_way(d, unit, time)
    denom = float(dt @ dt)
    if denom <= 0:
        return np.nan, np.nan, int(y.size)
    beta = float(dt @ yt) / denom
    resid = yt - beta * dt
    g = unit if cluster is None else np.asarray(cluster)[keep]
    order = np.argsort(g, kind="stable")
    gs, sc, rs = g[order], dt[order], resid[order]
    bounds = np.flatnonzero(np.r_[True, gs[1:] != gs[:-1], True])
    meat = 0.0
    for a, b in zip(bounds[:-1], bounds[1:]):
        meat += float(sc[a:b] @ rs[a:b]) ** 2
    ng = len(bounds) - 1
    if ng < 2:
        return beta, np.nan, int(y.size)
    adj = ng / (ng - 1.0)
    return beta, float(np.sqrt(adj * meat) / denom), int(y.size)


def event_study(y, evt, unit, time, leads=12, lags=12, ref=-1):
    """Coefficients by event time. `evt` is months since treatment; use a large
    sentinel (e.g. 9999) for never-treated units so they act as controls.

    Returns {event_time: (beta, se)} with `ref` omitted as the base period.
    """
    y = np.asarray(y, float); evt = np.asarray(evt)
    out = {}
    for k in range(-leads, lags + 1):
        if k == ref:
            continue
        d = (evt == k).astype(float)
        # compare period k against the reference period and the never-treated
        mask = np.isfinite(y) & ((evt == k) | (evt == ref) | (evt > 9000))
        if d[mask].sum() < 5:
            out[k] = (np.nan, np.nan); continue
        b, se, _ = twfe_did(y[mask], d[mask], unit[mask], time[mask])
        out[k] = (b, se)
    return out


def placebo_se(y, unit, time, n_treated, cohort_months, rng, n_perm=200,
               min_post=6):
    """Randomisation inference on never-treated units.

    Assign `n_treated` units a fake adoption month drawn from the real cohort
    distribution, estimate the DiD, repeat. The spread of the resulting
    estimates IS the standard error of this design at this sample size - no
    distributional assumption, no guessed sigma. The centre doubles as a
    placebo test: it should sit on zero.
    """
    y = np.asarray(y, float)
    units = np.unique(unit)
    if len(units) <= n_treated or len(cohort_months) == 0:
        return np.nan, np.nan, 0
    tmax = time.max()
    est = []
    for _ in range(n_perm):
        chosen = rng.choice(units, size=n_treated, replace=False)
        gmap = {u: rng.choice(cohort_months) for u in chosen}
        d = np.zeros(y.size)
        sel = np.isin(unit, chosen)
        gs = np.array([gmap.get(u, 10**9) for u in unit[sel]])
        d[sel] = (time[sel] >= gs).astype(float)
        if d.sum() < min_post:
            continue
        b, _, _ = twfe_did(y, d, unit, time)
        if np.isfinite(b):
            est.append(b)
    if len(est) < 20:
        return np.nan, np.nan, len(est)
    est = np.array(est)
    return float(est.mean()), float(est.std(ddof=1)), len(est)


def mde_from_se(se, alpha=0.05, power=0.80):
    """Smallest effect detectable at the given power. 1.96 + 0.84 for the
    conventional 5% / 80%."""
    return np.nan if not np.isfinite(se) else float(2.802 * se)


In [ ]:
%%writefile /content/cs_did.py
"""
Callaway & Sant'Anna (2021) group-time average treatment effects.

Why not two-way fixed effects: with staggered adoption and effects that differ
across cohorts, TWFE implicitly uses already-treated units as controls for
later adopters and weights the resulting comparisons with weights that can be
negative. The estimate can then sit outside the range of every underlying
effect, sign included. CS avoids this by never comparing a treated unit to
another treated unit.

Implemented for the case this study needs - a never-treated comparison group
and no covariates - where the doubly-robust estimator collapses to a
difference in mean changes:

    ATT(g,t) = E[Y_t - Y_{g-1} | G = g] - E[Y_t - Y_{g-1} | never treated]

Inference is the multiplier bootstrap over unit-level influence functions,
which clusters at the unit by construction and gives simultaneous bands.
"""
from __future__ import annotations
import numpy as np

__all__ = ["CSResult", "att_gt", "aggregate_dynamic", "aggregate_overall",
           "pretrend_test"]


class CSResult:
    def __init__(self, cohorts, periods, att, inf, n_g):
        self.cohorts, self.periods = cohorts, periods
        self.att = att          # (n_cohorts, n_periods)
        self.inf = inf          # (n_units, n_cohorts, n_periods)
        self.n_g = n_g          # treated units per cohort


def att_gt(Y, G, never_value=10**8, min_cell=5, max_lead=12, max_lag=24):
    """Group-time ATTs.

    Y : (n_units, n_periods) float array, NaN where unobserved
    G : (n_units,) integer column index of first treatment; `never_value` for
        never-treated units.

    Only event times in [-max_lead-1, +max_lag] are computed. That bounds the
    influence-function array - which is (units x cohorts x periods) and would
    otherwise run to half a gigabyte per outcome - and it makes the overall
    estimand explicit: the average effect over the `max_lag` months following
    adoption, rather than an average over whatever window each cohort happens
    to have left before the panel ends.
    """
    Y = np.asarray(Y, float); G = np.asarray(G)
    n, T = Y.shape
    never = G >= never_value
    cohorts = np.array(sorted({int(g) for g in G[~never] if 1 <= g <= T - 1}))
    periods = np.arange(T)
    att = np.full((len(cohorts), T), np.nan)
    inf = np.zeros((n, len(cohorts), T), dtype=np.float32)
    n_g = np.zeros(len(cohorts), int)

    for gi, g in enumerate(cohorts):
        base = g - 1
        in_g = (G == g)
        n_g[gi] = int(in_g.sum())
        if n_g[gi] < min_cell:
            continue
        lo, hi = max(0, g - max_lead - 1), min(T - 1, g + max_lag)
        for t in range(lo, hi + 1):
            if t == base:
                continue
            dY = Y[:, t] - Y[:, base]
            ok = np.isfinite(dY)
            tr = ok & in_g
            ct = ok & never
            n1, n0 = int(tr.sum()), int(ct.sum())
            if n1 < min_cell or n0 < min_cell:
                continue
            m1, m0 = dY[tr].mean(), dY[ct].mean()
            att[gi, t] = m1 - m0
            # influence function: mean-zero within each group by construction
            inf[tr, gi, t] = ((dY[tr] - m1) / n1).astype(np.float32)
            inf[ct, gi, t] = (-(dY[ct] - m0) / n0).astype(np.float32)
    return CSResult(cohorts, periods, att, inf, n_g)


def _boot(theta_inf, n_boot, rng):
    """Multiplier bootstrap with Rademacher weights, clustered at the unit."""
    n = theta_inf.shape[0]
    draws = np.empty((n_boot, theta_inf.shape[1]))
    for b in range(n_boot):
        v = rng.choice(np.array([-1.0, 1.0]), size=n)
        draws[b] = v @ theta_inf
    return draws


def aggregate_dynamic(res, e_min=-12, e_max=12, n_boot=500, rng=None,
                      weight_by_size=True):
    """Event-study aggregation: average ATT(g, g+e) across cohorts."""
    rng = rng or np.random.default_rng(0)
    es = np.arange(e_min, e_max + 1)
    n = res.inf.shape[0]
    theta = np.full(len(es), np.nan)
    theta_inf = np.zeros((n, len(es)))
    for k, e in enumerate(es):
        num, wsum, ifs = 0.0, 0.0, np.zeros(n)
        for gi, g in enumerate(res.cohorts):
            t = g + e
            if t < 0 or t >= res.att.shape[1] or not np.isfinite(res.att[gi, t]):
                continue
            w = res.n_g[gi] if weight_by_size else 1.0
            num += w * res.att[gi, t]; wsum += w
            ifs += w * res.inf[:, gi, t]
        if wsum > 0:
            theta[k] = num / wsum
            theta_inf[:, k] = ifs / wsum
    draws = _boot(theta_inf, n_boot, rng)
    se = draws.std(axis=0, ddof=1)
    # simultaneous band: sup-t critical value
    with np.errstate(invalid="ignore", divide="ignore"):
        tstat = np.nanmax(np.abs(draws / np.where(se > 0, se, np.nan)), axis=1)
    crit = float(np.nanquantile(tstat, 0.95)) if np.isfinite(tstat).any() else np.nan
    return {"event_time": es, "att": theta, "se": se, "crit_simultaneous": crit}


def aggregate_overall(res, n_boot=500, rng=None):
    """One number: average post-treatment effect, cohort-size weighted."""
    rng = rng or np.random.default_rng(0)
    n = res.inf.shape[0]
    num, wsum, ifs = 0.0, 0.0, np.zeros(n)
    for gi, g in enumerate(res.cohorts):
        post = [t for t in range(g, res.att.shape[1]) if np.isfinite(res.att[gi, t])]
        if not post:
            continue
        a = float(np.mean([res.att[gi, t] for t in post]))
        i = res.inf[:, gi, post].mean(axis=1)
        w = res.n_g[gi]
        num += w * a; wsum += w; ifs += w * i
    if wsum == 0:
        return {"att": np.nan, "se": np.nan, "n_treated": 0, "n_cohorts": 0}
    theta = num / wsum
    draws = _boot((ifs / wsum)[:, None], n_boot, rng)
    return {"att": float(theta), "se": float(draws.std(ddof=1)),
            "n_treated": int(res.n_g.sum()),
            "n_cohorts": int(sum(1 for x in res.n_g if x >= 1))}


def pretrend_test(res, e_min=-12, n_boot=500, rng=None):
    """Joint test that every pre-treatment ATT(g,t) is zero.

    The naive statistic - sum of squared z-scores against a chi-square - is
    WRONG here, and badly so. Every pre-period ATT shares the same base period
    g-1, so one noisy draw at the base shifts them all together; the estimates
    are strongly positively correlated and the sum of squares has nothing like
    a chi-square null. On synthetic data with no pre-trend at all, that
    statistic returned 35.3 on 11 df.

    The correct statistic is the maximum absolute z across pre-periods,
    compared against a critical value taken from the multiplier bootstrap of
    the same influence functions - which carries the correlation with it.
    This is the sup-t band that Callaway and Sant'Anna report.
    """
    rng = rng or np.random.default_rng(0)
    dyn = aggregate_dynamic(res, e_min=e_min, e_max=-2, n_boot=n_boot, rng=rng)
    ok = np.isfinite(dyn["att"]) & np.isfinite(dyn["se"]) & (dyn["se"] > 0)
    if ok.sum() == 0:
        return {"max_abs_z": np.nan, "crit": np.nan, "reject": None, "k": 0}
    z = dyn["att"][ok] / dyn["se"][ok]
    crit = dyn["crit_simultaneous"]
    return {"max_abs_z": float(np.max(np.abs(z))),
            "crit": float(crit),
            "reject": bool(np.max(np.abs(z)) > crit) if np.isfinite(crit) else None,
            "k": int(ok.sum()),
            "naive_sum_z2": float(np.sum(z ** 2)),
            "event_time": dyn["event_time"][ok], "att": dyn["att"][ok],
            "se": dyn["se"][ok]}


In [ ]:
import importlib, peltzman_io, panel_tools, cs_did
for m in (peltzman_io, panel_tools, cs_did): importlib.reload(m)
from peltzman_io import Job, atomic_write_text
from panel_tools import month_index, twfe_did
from cs_did import att_gt, aggregate_dynamic, aggregate_overall

# Per-key RNG, not a threaded one. A resumable job may execute fits in any
# order across sessions; seeding from the key makes every fit reproducible
# regardless of what ran before it.
def rng_for(key):
    h = int(hashlib.sha1(key.encode()).hexdigest()[:8], 16)
    return np.random.default_rng(SEED + h)
print("modules loaded")

## 1 · Rebuild the raw (bot-inclusive) outcomes

The human-only arm is loaded as-is from `nb4_panel_balanced.parquet`, so it is
bit-identical to what produced the published results — no chance of the contrast
being contaminated by an accidental change in sample or filter.

The raw arm re-runs NB4's outcome construction on the same balanced grid with
every `_h` column swapped for its all-actor counterpart. The grid, the zero-fill,
the `MIN_PRS` denominator floor and the cohort assignment are untouched.

In [ ]:
bal = pd.read_parquet(Path(STATE,"nb4_panel_balanced.parquet"))
raw = pd.read_parquet(Path(STATE,"nb3_panel.parquet"))
raw["t"] = raw.month.map(month_index)

NEED = ["prs_opened","prs_merged","merged_attributable","reviewers","review_comments",
        "reviews","prs_reviewed","changes_req","churn_merged","self_merged","pushes_default"]
missing = [c for c in NEED if c not in raw.columns]
assert not missing, f"nb3_panel.parquet is missing {missing} — it is not the post-NB3b panel"
assert "prs_opened_h" in raw.columns, ("nb3_panel.parquet has no human-only columns; "
    "re-run NB3b before this notebook")

d = bal.merge(raw[["repo","t"]+NEED], on=["repo","t"], how="left")
assert len(d) == len(bal), "merge changed the row count — nb3_panel has duplicate repo-months"
d[NEED] = d[NEED].fillna(0.0)     # same zero-fill NB4 applied to the balanced grid
del raw; gc.collect()             # 1.4M rows we no longer need

# --- raw outcomes: identical formulas, all-actor columns -------------------
den_pr  = d.prs_opened.where(d.prs_opened >= MIN_PRS)
den_mrg = d.prs_merged.where(d.prs_merged >= MIN_PRS)
den_att = d.merged_attributable.where(d.merged_attributable >= MIN_PRS)

d["raw_o1_reviewers_per_pr"]   = d.reviewers       / den_pr
d["raw_o2_comments_per_pr"]    = d.review_comments / den_pr
d["raw_o2b_reviews_per_pr"]    = d.reviews         / den_pr
d["raw_o3_review_coverage"]    = (d.prs_reviewed   / den_pr).clip(upper=1.0)
d["raw_o3b_changes_req_rate"]  = d.changes_req     / den_pr
d["raw_o4_churn_per_merged"]   = np.log1p(d.churn_merged / den_mrg)
d["raw_o5_self_merge_rate"]    = d.self_merged     / den_att
d["raw_o6_direct_push_share"]  = d.pushes_default  / (d.pushes_default + den_mrg)

OUTCOMES = [c for c in bal.columns if c.startswith(("o1_","o2_","o2b","o3_","o3b","o4_","o5_","o6_"))]
print(f"panel {len(d):,} rows · {d.repo.nunique():,} repos · {d.tt.nunique()} periods\n")
print(f"{'outcome':24s} {'human mean':>12s} {'raw mean':>12s} {'raw/human':>10s}")
for oc in OUTCOMES:
    h, r = d[oc].mean(), d["raw_"+oc].mean()
    print(f"{oc:24s} {h:12.4f} {r:12.4f} {r/h if h else np.nan:10.3f}")

In [ ]:
# matrix form, shared by both arms
units=np.sort(d.u.unique()); uidx={u:i for i,u in enumerate(units)}
T=int(d.tt.max())+1; N=len(units)
ui=d.u.map(uidx).to_numpy(); ti=d.tt.to_numpy(); TMIN=int(d.t.min())
NEVER=10**9

def as_matrix(col):
    M=np.full((N,T),np.nan); M[ui,ti]=d[col].to_numpy(); return M

g=d.groupby("u")["g_A"].min()
G_A=np.full(N,10**8)
for u,gv in g.items():
    if gv < NEVER: G_A[uidx[u]] = int(gv)-TMIN
_tr=G_A[G_A<10**8]
assert ((_tr>=0)&(_tr<T)).all(), "cohort scale mismatch"
print(f"matrix {N:,} x {T} · Treatment A: {len(_tr):,} treated in "
      f"{len(np.unique(_tr))} cohorts · {int((G_A>=10**8).sum()):,} never-treated")

## 2 · Estimate both arms

Sixteen fits: eight outcomes × {human-only, bot-inclusive}, each producing a
Callaway–Sant'Anna ATT and a TWFE coefficient. Checkpointed one fit per batch.

In [ ]:
KEYS = [f"{arm}|{oc}" for arm in ("human","raw") for oc in OUTCOMES]

def fit(key):
    arm, oc = key.split("|")
    col = oc if arm == "human" else "raw_" + oc
    Y  = as_matrix(col); sd = float(np.nanstd(Y))
    r  = att_gt(Y, G_A, min_cell=MIN_CELL, max_lead=abs(E_MIN), max_lag=E_MAX)
    ov = aggregate_overall(r, n_boot=N_BOOT, rng=rng_for(key))
    m  = d[col].notna().values
    bt, st, _ = twfe_did(d[col].values[m], d["post_A"].values[m],
                         d.u.values[m], d.tt.values[m])
    del r, Y; gc.collect()
    t = ov["att"]/ov["se"] if ov["se"] else np.nan
    print(f"  {arm:6s} {oc:24s} CS {ov['att']:+9.4f} ± {ov['se']:.4f} "
          f"({ov['att']/sd if sd else np.nan:+.3f} sd)   TWFE {bt:+9.4f}", flush=True)
    return [{"arm":arm,"outcome":oc,"cs_att":ov["att"],"cs_se":ov["se"],"t":t,
             "att_in_sd":ov["att"]/sd if sd else np.nan,"twfe":bt,"twfe_se":st,
             "n_treated":ov["n_treated"],"outcome_sd":sd}]

job = Job("nb6_botcontrast", keys=KEYS, root=STATE, batch_size=1,
          params={"n_boot":N_BOOT,"min_cell":MIN_CELL,"e":[E_MIN,E_MAX],"seed":SEED})
job.run(fit, on_error="raise", progress_every=2)
B = job.frame()
print(f"\n{len(B)} fits complete")

In [ ]:
W = (B[B.arm=="human"].set_index("outcome")
       .join(B[B.arm=="raw"].set_index("outcome"), lsuffix="_h", rsuffix="_r"))
W["cs_shift_sd"]  = W.att_in_sd_h - W.att_in_sd_r
W = W.reindex([o for o in OUTCOMES if o in W.index])

print("Bot contamination · Treatment A · same sample, same estimator\n")
print(f"{'outcome':24s} {'CS raw':>10s} {'CS human':>10s} | {'raw sd':>8s} {'human sd':>9s} "
      f"{'shift':>8s} {'flip':>5s}")
for oc, r in W.iterrows():
    flip = "YES" if np.isfinite(r.cs_att_h*r.cs_att_r) and r.cs_att_h*r.cs_att_r < 0 else ""
    print(f"{oc:24s} {r.cs_att_r:+10.4f} {r.cs_att_h:+10.4f} | {r.att_in_sd_r:+8.3f} "
          f"{r.att_in_sd_h:+9.3f} {r.cs_shift_sd:+8.3f} {flip:>5s}")

print("\nTWFE, same contrast")
for oc, r in W.iterrows():
    print(f"{oc:24s} raw {r.twfe_r:+9.4f}   human {r.twfe_h:+9.4f}")

n_flip = int(((W.cs_att_h*W.cs_att_r) < 0).sum())
print(f"\nsign flips induced by bot exclusion: {n_flip} of {len(W)}")
print(f"mean |shift| in sd units: {W.cs_shift_sd.abs().mean():.3f}")

W.reset_index().to_csv(OUT/"results_bots.csv", index=False)
print(f"\nwrote {OUT}/results_bots.csv")

In [ ]:
# reconciliation: the human arm must reproduce NB5 to within bootstrap noise
nb5 = json.loads(Path(STATE,"nb5_summary.json").read_text())["treatment_A"]
print("human arm vs published NB5 results\n")
print(f"{'outcome':24s} {'NB5 ATT':>10s} {'NB6 ATT':>10s} {'diff':>9s} {'diff/se':>8s}")
worst = 0.0
for oc, r in W.iterrows():
    if oc not in nb5: continue
    dlt = r.cs_att_h - nb5[oc]["att"]
    z = abs(dlt)/r.cs_se_h if r.cs_se_h else np.nan
    worst = max(worst, z if np.isfinite(z) else 0)
    print(f"{oc:24s} {nb5[oc]['att']:+10.4f} {r.cs_att_h:+10.4f} {dlt:+9.5f} {z:8.3f}")
print(f"\nlargest discrepancy: {worst:.3f} SE")
print("\nThe CS point estimate is deterministic given the panel — only the standard\n"
      "error comes from the bootstrap. So these should agree to floating-point, and\n"
      "any visible difference means the sample or the cohort assignment has moved.")
print("VERDICT:", "reproduces NB5 exactly — the contrast is trustworthy" if worst < 0.05
      else "DOES NOT REPRODUCE NB5 — stop and investigate before using this table")

## 3 · The two missing event studies

`event_study.csv` covers the primary outcome, review coverage and self-merge rate.
Comments per PR and direct-push share also survive Benjamini–Hochberg, so the
four-panel figure in §5 needs them too.

In [ ]:
MISSING = ["o2_comments_per_pr","o6_direct_push_share"]

def es_fit(key):
    oc = key
    r  = att_gt(as_matrix(oc), G_A, min_cell=MIN_CELL, max_lead=abs(E_MIN), max_lag=E_MAX)
    dy = aggregate_dynamic(r, E_MIN, E_MAX, n_boot=N_BOOT, rng=rng_for("es|"+key))
    del r; gc.collect()
    post=[a for e,a in zip(dy["event_time"],dy["att"]) if e>=0 and np.isfinite(a)]
    print(f"  {oc:24s} mean post ATT {np.mean(post):+.4f} · "
          f"simultaneous crit {dy['crit_simultaneous']:.2f}", flush=True)
    return [{"outcome":oc,"event_time":int(e),"att":float(a),"se":float(s),
             "crit":float(dy["crit_simultaneous"])}
            for e,a,s in zip(dy["event_time"],dy["att"],dy["se"])]

esjob = Job("nb6_eventstudy", keys=MISSING, root=STATE, batch_size=1,
            params={"n_boot":N_BOOT,"min_cell":MIN_CELL,"e":[E_MIN,E_MAX],"seed":SEED})
esjob.run(es_fit, on_error="raise")
E = esjob.frame()

old = pd.read_csv(OUT/"event_study.csv")
wide = E.pivot(index="event_time", columns="outcome", values=["att","se"])
wide.columns = [f"{oc}_{k}" for k,oc in wide.columns]
merged = old.merge(wide.reset_index(), on="event_time", how="outer").sort_values("event_time")
ocs = [c[:-4] for c in merged.columns if c.endswith("_att")]
merged = merged[["event_time"] + [f"{o}_{k}" for o in ocs for k in ("att","se")]]
merged.to_csv(OUT/"event_study.csv", index=False)
print(f"\nevent_study.csv now covers "
      f"{len([c for c in merged.columns if c.endswith('_att')])} outcomes")
print(merged.head(3).to_string(index=False))

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
PANELS=[PRIMARY,"o3_review_coverage","o5_self_merge_rate",
        "o2_comments_per_pr","o6_direct_push_share"]
have=[p for p in PANELS if f"{p}_att" in merged.columns]
fig,axes=plt.subplots(1,len(have),figsize=(3.4*len(have),3.6),sharex=True)
for ax,oc in zip(np.atleast_1d(axes),have):
    e=merged.event_time.values; a=merged[f"{oc}_att"].values; s=merged[f"{oc}_se"].values
    ax.axhline(0,color="#888",lw=.8); ax.axvline(-0.5,color="#b00",lw=.8,ls="--")
    ax.errorbar(e,a,yerr=1.96*s,fmt="o-",ms=2.5,lw=.9,capsize=1.5)
    ax.set_title(oc.replace("_"," "),fontsize=9); ax.set_xlabel("months since adoption")
fig.tight_layout(); fig.savefig(FIG/"nb6_event_studies.png",dpi=150)
print("figure ->",FIG/"nb6_event_studies.png")

summary={"n_fits":int(len(B)),"n_sign_flips_from_bots":n_flip,
         "mean_abs_shift_sd":float(W.cs_shift_sd.abs().mean()),
         "nb5_max_discrepancy_se":float(worst),
         "event_study_outcomes":[c[:-4] for c in merged.columns if c.endswith("_att")],
         "bot_contrast":{oc:{"cs_raw":float(r.cs_att_r),"cs_human":float(r.cs_att_h),
                             "sd_raw":float(r.att_in_sd_r),"sd_human":float(r.att_in_sd_h)}
                         for oc,r in W.iterrows()}}
atomic_write_text(Path(STATE,"nb6_summary.json"),json.dumps(summary,indent=1,default=float))
print("\n"+json.dumps(summary,indent=1,default=float))